<a href="https://colab.research.google.com/github/Ryan56-56/project1ML/blob/main/monday.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import imageio.v2 as imageio
from tqdm import tqdm

In [2]:
train_transform = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(),
    T.RandomCrop(32, padding=4),
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std=[0.2470, 0.2435, 0.2616])
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.ToTensor(),
    T.Normalize(mean=[0.4914, 0.4822, 0.4465],
                std=[0.2470, 0.2435, 0.2616])
])


In [3]:
class CIFARFolderDatasetRAM(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.data = []
        self.labels = []
        self.classes = sorted(os.listdir(root))

        print("Loading dataset into RAM...")

        for label_idx, folder in enumerate(self.classes):
            folder_path = os.path.join(root, folder)
            for img_name in os.listdir(folder_path):
                img = imageio.imread(os.path.join(folder_path, img_name))
                self.data.append(img)
                self.labels.append(label_idx)

        print(f"Loaded {len(self.data)} images into RAM.")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = self.data[idx]
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

In [8]:
train_path = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/CIFAR-10-images-master/train"
test_path  = "/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/test"

train_ds = CIFARFolderDatasetRAM(train_path, transform=train_transform)
test_ds  = CIFARFolderDatasetRAM(test_path, transform=test_transform)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,
                      num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds, batch_size=256, shuffle=False,
                      num_workers=2, pin_memory=True)

Loading dataset into RAM...


KeyboardInterrupt: 

In [7]:
import os
print(os.listdir("/content/drive/MyDrive/CIFAR-10-images-master/CIFAR-10-images-master/"))

['test', 'README.md', 'CIFAR-10-images-master']


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class BetterCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

In [ ]:
def train_model(model, train_dl, loss_fn, opt, epochs, ckpt_path):
    model.train()
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(epochs):
        loop = tqdm(train_dl, desc=f"Epoch {epoch}", leave=False)
        running_loss = 0

        for xb, yb in loop:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            opt.zero_grad()

            with torch.cuda.amp.autocast():
                preds = model(xb)
                loss = loss_fn(preds, yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            running_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        print(f"Epoch {epoch} | Loss: {running_loss/len(train_dl):.4f}")

        if epoch % 5 == 0:
            torch.save(model.state_dict(), os.path.join(ckpt_path, f"epoch_{epoch}.pth"))


In [ ]:
def evaluate(model, test_dl):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in test_dl:
            xb = xb.to(device, non_blocking=True)
            preds = model(xb)
            _, predicted = torch.max(preds, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(yb.numpy())

    print("Accuracy:", accuracy_score(all_labels, all_preds))
    print("Precision:", precision_score(all_labels, all_preds, average='weighted'))
    print("Recall:", recall_score(all_labels, all_preds, average='weighted'))
    print("F1 Score:", f1_score(all_labels, all_preds, average='weighted'))


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BetterCNN().to(device)
model = torch.compile(model)

opt = optim.Adam(model.parameters(), lr=0.0005)
loss_fn = nn.CrossEntropyLoss()

CKPT = "/content/drive/MyDrive/CNN_model_CIFAR10"
os.makedirs(CKPT, exist_ok=True)

train_model(model, train_dl, loss_fn, opt, epochs=40, ckpt_path=CKPT)
